# Phase 1+2 Accuracy Tables

Preedit and judge accuracy against ground truth (y16).  
No GPU needed — runs on any node as soon as phase 1+2 CSVs exist.

In [1]:
import os, sys
import pandas as pd

PROJECT_ROOT = "/scratch/jq2uw/derm_vlms"
SIMEDIT = os.path.join(PROJECT_ROOT, "prelim_simedit")
if SIMEDIT not in sys.path:
    sys.path.insert(0, SIMEDIT)

from utils.eval import parse_top1, to_y16

RESULTS = os.path.join(SIMEDIT, "results_local")
ROBOTS = ["medgemma", "dermato_llama"]
JUDGES = ["gpt54", "claude_opus48", "ground_truth"]

In [2]:
# --- Preedit accuracy (same for all judges, one per robot) ---

STANDARD_Y16 = {
    "Actinic Keratosis", "Basal Cell Carcinoma", "Dermatofibroma",
    "Fibrous Papule", "Hemangioma", "Melanocytic Lesion",
    "Melanocytic Nevus", "Melanocytic Tumor", "Melanoma",
    "Seborrheic Keratosis", "Squamous Cell Carcinoma",
    "Squamous Cell Carcinoma In Situ",
}

def normalize_gt(x):
    return x if x in STANDARD_Y16 else "Other"

preedit_rows = []
for robot in ROBOTS:
    path = os.path.join(RESULTS, robot, "01_preedit__top_1.csv")
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    df["gt_y16"] = df["gt_y16"].apply(normalize_gt)
    df["preedit_y16"] = df["preedit_dx"].apply(to_y16)
    df["correct"] = df["preedit_y16"] == df["gt_y16"]
    preedit_rows.append({
        "robot": robot,
        "n": len(df),
        "preedit_acc": df["correct"].mean(),
    })

df_preedit = pd.DataFrame(preedit_rows)
print("=== Preedit Accuracy (top-1 vs GT) ===")
df_preedit

=== Preedit Accuracy (top-1 vs GT) ===


,robot,n,preedit_acc
0,medgemma,1031,0.331717
1,dermato_llama,1031,0.341416


In [3]:
# --- Judge accuracy ---

judge_rows = []
for robot in ROBOTS:
    for judge in JUDGES:
        path = os.path.join(RESULTS, robot, f"02_judge__{judge}__top_1.csv")
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        df["gt_y16"] = df["gt_y16"].apply(normalize_gt)
        df["judge_y16"] = df["judge_dx"].apply(to_y16)
        df["correct"] = df["judge_y16"] == df["gt_y16"]
        n_errors = (df["judge_dx"].isna() | (df["judge_dx"] == "")).sum()
        judge_rows.append({
            "robot": robot,
            "judge": judge,
            "n": len(df),
            "n_errors": int(n_errors),
            "judge_acc": df["correct"].mean(),
        })

df_judge = pd.DataFrame(judge_rows)
print("=== Judge Accuracy (top-1 vs GT) ===")
df_judge
print("=== Judge Accuracy (top-1 vs GT) ===")
df_judge

=== Judge Accuracy (top-1 vs GT) ===
=== Judge Accuracy (top-1 vs GT) ===


,robot,judge,n,n_errors,judge_acc
0,medgemma,gpt54,1031,14,0.311348
1,medgemma,claude_opus48,1031,0,0.376334
2,medgemma,ground_truth,1031,0,1.000000
3,dermato_llama,gpt54,1031,14,0.308438
4,dermato_llama,claude_opus48,1031,0,0.369544
5,dermato_llama,ground_truth,1031,0,1.000000


In [4]:
# --- Postedit accuracy ---

postedit_rows = []
for robot in ROBOTS:
    for judge in JUDGES:
        path = os.path.join(RESULTS, robot, f"03_postedit__{judge}__top_1.csv")
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        df["gt_y16"] = df["gt_y16"].apply(normalize_gt)
        df["postedit_y16"] = df["postedit_dx"].apply(to_y16)
        df["correct"] = df["postedit_y16"] == df["gt_y16"]
        n_errors = (df["postedit_dx"].isna() | (df["postedit_dx"] == "")).sum()
        postedit_rows.append({
            "robot": robot,
            "judge": judge,
            "n": len(df),
            "n_errors": int(n_errors),
            "postedit_acc": df["correct"].mean(),
        })

df_postedit = pd.DataFrame(postedit_rows)
print("=== Postedit Accuracy (top-1 vs GT) ===")
df_postedit

=== Postedit Accuracy (top-1 vs GT) ===


,robot,judge,n,n_errors,postedit_acc
0,medgemma,gpt54,1031,12,0.307468
1,medgemma,claude_opus48,1031,16,0.373424
2,medgemma,ground_truth,1031,0,1.000000
3,dermato_llama,gpt54,1031,5,0.307468
4,dermato_llama,claude_opus48,1031,24,0.368574
5,dermato_llama,ground_truth,1031,0,0.988361


In [5]:
# --- Combined summary: preedit → judge → postedit ---

summary = df_postedit[["robot", "judge", "n"]].copy()
summary = summary.merge(
    df_preedit[["robot", "preedit_acc"]], on="robot", how="left"
)
summary = summary.merge(
    df_judge[["robot", "judge", "judge_acc"]], on=["robot", "judge"], how="left"
)
summary = summary.merge(
    df_postedit[["robot", "judge", "postedit_acc"]], on=["robot", "judge"], how="left"
)
summary["delta"] = summary["postedit_acc"] - summary["preedit_acc"]

print("=== Full Pipeline Accuracy (top-1 vs GT) ===")
summary

=== Full Pipeline Accuracy (top-1 vs GT) ===


,robot,judge,n,preedit_acc,judge_acc,postedit_acc,delta
0,medgemma,gpt54,1031,0.331717,0.311348,0.307468,-0.024248
1,medgemma,claude_opus48,1031,0.331717,0.376334,0.373424,0.041707
2,medgemma,ground_truth,1031,0.331717,1.000000,1.000000,0.668283
3,dermato_llama,gpt54,1031,0.341416,0.308438,0.307468,-0.033948
4,dermato_llama,claude_opus48,1031,0.341416,0.369544,0.368574,0.027158
5,dermato_llama,ground_truth,1031,0.341416,1.000000,0.988361,0.646945
